# 📊 Data Exploration - MVTec AD Bottle Category

This notebook explores the MVTec Anomaly Detection dataset (Bottle category) to understand:
- Dataset structure and size
- Image properties (size, mode, channels)
- Normal vs defective image comparison
- Ground truth mask analysis
- Statistical summary

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

DATA_DIR = '../data/mvtec_anomaly_detection/bottle'
print('Dataset directory:', os.path.abspath(DATA_DIR))
print('Exists:', os.path.exists(DATA_DIR))

## 1. Dataset Structure Overview

In [ ]:
# Scan dataset structure
def scan_directory(root):
    """Recursively scan directory and count files."""
    summary = {}
    for dirpath, dirnames, filenames in os.walk(root):
        rel = os.path.relpath(dirpath, root)
        img_files = [f for f in filenames if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            summary[rel] = len(img_files)
    return summary

structure = scan_directory(DATA_DIR)
print(f"{'Directory':<40} {'Images':>10}")
print('-' * 52)
total = 0
for path, count in sorted(structure.items()):
    print(f"{path:<40} {count:>10}")
    total += count
print('-' * 52)
print(f"{'TOTAL':<40} {total:>10}")

## 2. Image Properties Analysis

In [ ]:
# Analyze image properties
train_good_dir = os.path.join(DATA_DIR, 'train', 'good')
train_images = sorted([f for f in os.listdir(train_good_dir) if f.endswith('.png')])

print(f'Total normal training images: {len(train_images)}')

# Check properties of first image
sample = Image.open(os.path.join(train_good_dir, train_images[0]))
print(f'Image size: {sample.size}')
print(f'Image mode: {sample.mode}')
print(f'Channels: {len(sample.getbands())}')

# Check all sizes
sizes = set()
modes = set()
for f in train_images:
    img = Image.open(os.path.join(train_good_dir, f))
    sizes.add(img.size)
    modes.add(img.mode)

print(f'\nUnique sizes: {sizes}')
print(f'Unique modes: {modes}')

## 3. Visualize Normal Training Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Normal Training Images (Bottle)', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(train_images):
        img = Image.open(os.path.join(train_good_dir, train_images[i]))
        ax.imshow(img)
        ax.set_title(train_images[i], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Visualize Defective Test Images

In [ ]:
test_dir = os.path.join(DATA_DIR, 'test')
gt_dir = os.path.join(DATA_DIR, 'ground_truth')

defect_types = [d for d in sorted(os.listdir(test_dir)) 
                if os.path.isdir(os.path.join(test_dir, d)) and d != 'good']
print('Defect types:', defect_types)

# Show samples from each defect type with ground truth masks
fig, axes = plt.subplots(len(defect_types), 4, figsize=(16, 4 * len(defect_types)))
if len(defect_types) == 1:
    axes = axes[np.newaxis, :]

fig.suptitle('Defect Types with Ground Truth Masks', fontsize=16, fontweight='bold')

for row, defect in enumerate(defect_types):
    defect_dir = os.path.join(test_dir, defect)
    defect_images = sorted([f for f in os.listdir(defect_dir) if f.endswith('.png')])
    
    if defect_images:
        # Show first defective image
        img_path = os.path.join(defect_dir, defect_images[0])
        img = Image.open(img_path)
        
        # Find corresponding mask
        mask_name = defect_images[0].replace('.png', '_mask.png')
        mask_path = os.path.join(gt_dir, defect, mask_name)
        
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{defect} - Original')
        axes[row, 0].axis('off')
        
        if os.path.exists(mask_path):
            mask = Image.open(mask_path)
            axes[row, 1].imshow(mask, cmap='gray')
            axes[row, 1].set_title(f'{defect} - Ground Truth')
        axes[row, 1].axis('off')
        
        # Show overlay
        if os.path.exists(mask_path):
            axes[row, 2].imshow(img)
            mask_arr = np.array(mask.resize(img.size))
            axes[row, 2].imshow(mask_arr, cmap='Reds', alpha=0.4)
            axes[row, 2].set_title(f'{defect} - Overlay')
        axes[row, 2].axis('off')
        
        # Show second sample if available
        if len(defect_images) > 1:
            img2 = Image.open(os.path.join(defect_dir, defect_images[1]))
            axes[row, 3].imshow(img2)
            axes[row, 3].set_title(f'{defect} - Sample 2')
        axes[row, 3].axis('off')

plt.tight_layout()
plt.show()

## 5. Normal vs Defective Side-by-Side

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Normal vs Defective Comparison', fontsize=16, fontweight='bold')

# Normal
normal_img = Image.open(os.path.join(train_good_dir, train_images[0]))
axes[0].imshow(normal_img)
axes[0].set_title('✅ NORMAL (PASS)', fontsize=14, color='green', fontweight='bold')
axes[0].axis('off')

# Defective samples
for i, defect in enumerate(defect_types[:3]):
    defect_dir = os.path.join(test_dir, defect)
    imgs = sorted([f for f in os.listdir(defect_dir) if f.endswith('.png')])
    if imgs:
        img = Image.open(os.path.join(defect_dir, imgs[0]))
        axes[i + 1].imshow(img)
        axes[i + 1].set_title(f'❌ {defect.upper()} (FAIL)', fontsize=14, color='red', fontweight='bold')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

## 6. Pixel Intensity Distribution

In [ ]:
# Compare pixel distributions of normal vs defective
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normal image histogram
normal_arr = np.array(normal_img)
for c, color in zip(range(3), ['red', 'green', 'blue']):
    axes[0].hist(normal_arr[:, :, c].flatten(), bins=50, alpha=0.5, color=color, label=color.upper())
axes[0].set_title('Normal Image - Pixel Distribution')
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Count')
axes[0].legend()

# Defective image histogram
if defect_types:
    defect_dir = os.path.join(test_dir, defect_types[0])
    defect_imgs = sorted([f for f in os.listdir(defect_dir) if f.endswith('.png')])
    if defect_imgs:
        defect_arr = np.array(Image.open(os.path.join(defect_dir, defect_imgs[0])))
        for c, color in zip(range(3), ['red', 'green', 'blue']):
            axes[1].hist(defect_arr[:, :, c].flatten(), bins=50, alpha=0.5, color=color, label=color.upper())
        axes[1].set_title(f'Defective ({defect_types[0]}) - Pixel Distribution')
        axes[1].set_xlabel('Pixel Value')
        axes[1].set_ylabel('Count')
        axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Dataset Summary Statistics

In [ ]:
# Count images per split
splits = {}

# Training
train_good = len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', 'good')) if f.endswith('.png')])
splits['train/good'] = train_good

# Test
test_good = len([f for f in os.listdir(os.path.join(DATA_DIR, 'test', 'good')) if f.endswith('.png')])
splits['test/good'] = test_good

for defect in defect_types:
    count = len([f for f in os.listdir(os.path.join(DATA_DIR, 'test', defect)) if f.endswith('.png')])
    splits[f'test/{defect}'] = count

print('\n📊 Dataset Summary')
print('=' * 40)
for split, count in splits.items():
    label = '✅ Normal' if 'good' in split else '❌ Defective'
    print(f'  {split:<25} {count:>5} images  ({label})')
print('=' * 40)
print(f'  {"Total":<25} {sum(splits.values()):>5} images')

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))
labels = list(splits.keys())
values = list(splits.values())
colors = ['#2ecc71' if 'good' in l else '#e74c3c' for l in labels]
ax.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Dataset Distribution', fontsize=14)
plt.show()

## 8. Ground Truth Mask Analysis

In [ ]:
# Analyze defect sizes from ground truth masks
print('Ground Truth Mask Analysis')
print('=' * 50)

for defect in defect_types:
    mask_dir = os.path.join(gt_dir, defect)
    if not os.path.exists(mask_dir):
        continue
    
    masks = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])
    defect_ratios = []
    
    for m in masks:
        mask = np.array(Image.open(os.path.join(mask_dir, m)).convert('L'))
        total_pixels = mask.size
        defect_pixels = (mask > 128).sum()
        ratio = defect_pixels / total_pixels * 100
        defect_ratios.append(ratio)
    
    print(f'\n{defect}:')
    print(f'  Masks: {len(masks)}')
    print(f'  Avg defect area: {np.mean(defect_ratios):.2f}%')
    print(f'  Min defect area: {np.min(defect_ratios):.2f}%')
    print(f'  Max defect area: {np.max(defect_ratios):.2f}%')

## ✅ Summary

**Key findings:**
- All images are consistent in size and format
- Training data contains only normal (good) images
- Multiple defect categories available for evaluation
- Ground truth masks provide pixel-level supervision for localization metrics
- Defect regions are typically small relative to the total image area

**Next steps:**
1. Build preprocessing pipeline (resize to 256×256, convert to tensors)
2. Create PyTorch Dataset and DataLoader
3. Train the Convolutional Autoencoder